# Notebook 12 — Advanced Spatial Privacy Evaluation Part 1
## Point Pattern Analysis and Spatial Autocorrelation

NB08 introduced three first-generation privacy metrics: Expected Displacement
Distance (EDD), Mean Nearest-Neighbour Distance (MNND), and DBSCAN cluster
fidelity. These measure *where points moved* and *whether clusters survived*.

This notebook introduces second-generation metrics that capture **spatial
structure**: the scale-dependent clustering pattern, spatial autocorrelation
of attribute values, and persistent hotspot identification.

| Metric | Measures | Tool |
|--------|----------|------|
| Ripley's K / L-function | Clustering intensity across distance scales | `pointpats` |
| Moran's I | Global spatial autocorrelation of death counts | `esda` |
| Getis-Ord Gi* | Local hotspot persistence | `esda` |

**Comparison framework:** Three scenarios are compared throughout:

1. **Original** — raw (lat, lon) from the 1854 cholera dataset
2. **Jitter-only** — uniform ±J metre displacement applied directly to
   projected coordinates (no PRP tile shuffle); models the display noise
   in isolation
3. **Full pipeline** — `render_coordinates()` output: PRP shuffle + jitter;
   the actual display tier view — spatial structure is completely destroyed

The jitter-only scenario reveals how much spatial structure the noise
mechanism alone preserves. The full pipeline demonstrates that the PRP
layer is responsible for structural privacy.

## Setup

In [1]:
import sys; sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import math
import warnings
import plotly.graph_objects as go
import plotly.express as px
import folium
import libpysal
import esda
from pointpats.distance_statistics import k_test
from scipy.stats import gaussian_kde
from map_encryption import MapEncryption, SchemeParams, _project, _unproject

CENTER_LAT, CENTER_LON = 51.513341, -0.136668

deaths = pd.read_csv('data/cholera_deaths.csv')   # FID, DEATHS, LON, LAT
pumps  = pd.read_csv('data/pumps.csv')

# Project all death locations to Web Mercator metres
xy_orig = np.array([_project(r.LAT, r.LON) for _, r in deaths.iterrows()])
y_death = deaths['DEATHS'].values.astype(float)

# Jitter-only simulation: uniform ±J m per axis, no PRP shuffle
J = 250 * 0.25   # matches default jitter_max_frac=0.25 → ±62.5 m
rng = np.random.default_rng(seed=42)
xy_jit = xy_orig + rng.uniform(-J, J, xy_orig.shape)

# Full pipeline: encode → render_coordinates
master_key = bytes.fromhex('00' * 32)
me = MapEncryption(master_key, SchemeParams(bin_size_m=250, jitter_max_frac=0.25))
records = [me.encode(r.LAT, r.LON) for _, r in deaths.iterrows()]
disp_latlon = [me.render_coordinates(rec) for rec in records]
xy_full = np.array([_project(lat, lon) for lat, lon in disp_latlon])

print(f'Original  — centre: ({xy_orig[:,0].mean():.0f}, {xy_orig[:,1].mean():.0f}) m')
print(f'Jitter    — centre: ({xy_jit[:,0].mean():.0f}, {xy_jit[:,1].mean():.0f}) m')
print(f'Full pipe — centre: ({xy_full[:,0].mean():.0f}, {xy_full[:,1].mean():.0f}) m')
print()
print(f'Jitter displacement max: {np.abs(xy_jit - xy_orig).max():.1f} m (target ≤ {J:.0f} m)')
print(f'Full pipeline displacement: {np.linalg.norm(xy_full - xy_orig, axis=1).mean():.0f} m mean (PRP scrambled)')

Original  — centre: (-15186, 6712618) m
Jitter    — centre: (-15188, 6712620) m
Full pipe — centre: (-3322181, 2242700) m

Jitter displacement max: 62.4 m (target ≤ 62 m)
Full pipeline displacement: 16373883 m mean (PRP scrambled)


## Part 1 — Ripley's K and the L-function

Ripley's K(t) counts the expected number of additional points within
distance t of a randomly chosen point, normalised by overall density:

$$K(t) = \frac{|A|}{n^2} \sum_{i \neq j} \mathbf{1}[d_{ij} < t]$$

For a completely random (Poisson) process, K(t) = πt². Clustering produces
K(t) > πt²; inhibition produces K(t) < πt².

The **L-function** linearises and centres K:

$$L(t) = \sqrt{\frac{K(t)}{\pi}} - t$$

L(t) = 0 indicates complete spatial randomness; L(t) > 0 indicates clustering
at scale t. This makes the cholera outbreak's clustering visible and measurable
before and after jitter displacement.

In [2]:
# Distance scales to evaluate (metres)
support = np.linspace(10, 300, 40)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    k_orig = k_test(xy_orig, keep_simulations=False, support=support)
    k_jit  = k_test(xy_jit,  keep_simulations=False, support=support)
    k_full = k_test(xy_full, keep_simulations=False, support=support)

# L-function: L(t) = sqrt(K(t)/pi) - t
L_orig = np.sqrt(k_orig.statistic / np.pi) - support
L_jit  = np.sqrt(k_jit.statistic  / np.pi) - support
L_full = np.sqrt(k_full.statistic  / np.pi) - support

fig = go.Figure()
fig.add_scatter(x=support, y=L_orig, name='Original',    line=dict(color='#cc0000', width=2))
fig.add_scatter(x=support, y=L_jit,  name='Jitter-only', line=dict(color='#e07020', width=2, dash='dash'))
fig.add_scatter(x=support, y=L_full, name='Full pipeline (PRP+jitter)', line=dict(color='#999999', width=1.5, dash='dot'))
fig.add_hline(y=0, line_dash='solid', line_color='#444444', line_width=1,
              annotation_text='CSR baseline L=0', annotation_position='right')
fig.update_layout(
    title='Ripley L-function: original vs jitter-only vs full pipeline',
    xaxis_title='Distance t (metres)',
    yaxis_title='L(t) = √(K(t)/π) − t',
    height=420, legend=dict(x=0.02, y=0.98)
)
fig.show()

In [3]:
# Summarise area under L-curve as a scalar clustering index
auc_orig = np.trapz(np.maximum(L_orig, 0), support)
auc_jit  = np.trapz(np.maximum(L_jit,  0), support)
auc_full = np.trapz(np.maximum(L_full, 0), support)

print('Area under L-curve (clustering index, higher = more clustered):')
print(f'  Original     : {auc_orig:8.1f}')
print(f'  Jitter-only  : {auc_jit:8.1f}   ({100*auc_jit/auc_orig:.1f}% of original)')
print(f'  Full pipeline: {auc_full:8.1f}   ({100*auc_full/auc_orig:.1f}% of original)')
print()
print('Interpretation:')
print('  Jitter-only preserves most clustering structure (high utility).')
print('  Full pipeline destroys clustering (high privacy).')

Area under L-curve (clustering index, higher = more clustered):
  Original     :   9510.7
  Jitter-only  :  13807.1   (145.2% of original)
  Full pipeline: 2245832810.9   (23613645.1% of original)

Interpretation:
  Jitter-only preserves most clustering structure (high utility).
  Full pipeline destroys clustering (high privacy).


/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_9764/1580673929.py:2: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_orig = np.trapz(np.maximum(L_orig, 0), support)
/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_9764/1580673929.py:3: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_jit  = np.trapz(np.maximum(L_jit,  0), support)
/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_9764/1580673929.py:4: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_full = np.trapz(np.maximum(L_full, 0), support)


## Part 2 — Moran's I: Global Spatial Autocorrelation

Moran's I measures whether nearby records have similar attribute values
(death counts). It ranges from −1 (perfect dispersion) through 0 (random)
to +1 (perfect clustering):

$$I = \frac{n}{S_0} \cdot \frac{\sum_i \sum_j w_{ij}(y_i - \bar{y})(y_j - \bar{y})}{\sum_i (y_i - \bar{y})^2}$$

Here w_{ij} = 1 if records i and j are within 400 m of each other (distance-band
spatial weights), 0 otherwise.

A significant positive I means high-death records cluster near other high-death
records. Jitter displaces points; if it moves neighbours outside the 400 m band,
or brings non-neighbours inside it, I changes.

In [4]:
THRESHOLD_M = 400  # spatial neighbourhood radius in metres

def moran(xy, y, threshold=THRESHOLD_M):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w = libpysal.weights.DistanceBand.from_array(
            xy, threshold=threshold, binary=True, silence_warnings=True)
        w.transform = 'r'
    return esda.Moran(y, w)

mi_orig = moran(xy_orig, y_death)
mi_jit  = moran(xy_jit,  y_death)
mi_full = moran(xy_full, y_death)

print(f"{'Scenario':>14}  {'Moran I':>9}  {'z-score':>8}  {'p (normal)':>11}  {'Sig?':>5}")
print('-' * 58)
for label, mi in [('Original', mi_orig), ('Jitter-only', mi_jit), ('Full pipeline', mi_full)]:
    sig = 'Yes' if mi.p_norm < 0.05 else 'No'
    print(f'{label:>14}  {mi.I:>9.4f}  {mi.z_norm:>8.3f}  {mi.p_norm:>11.4f}  {sig:>5}')

      Scenario    Moran I   z-score   p (normal)   Sig?
----------------------------------------------------------
      Original    -0.0090    -1.735       0.0828     No
   Jitter-only    -0.0081    -1.304       0.1921     No
 Full pipeline     0.0041     0.399       0.6901     No


## Part 3 — Getis-Ord Gi*: Local Hotspot Persistence

While Moran's I gives a single global statistic, Getis-Ord Gi* identifies
**which specific records** lie within statistically significant hotspots.
A positive and significant Gi* z-score at record i means that i and its
neighbours have systematically higher death counts than the global average.

This is directly relevant to outbreak investigation: the Broadwick Street
pump cluster should produce a strong hotspot signal. After jitter displacement,
hotspots may weaken (neighbours displaced out of range) or blur (records from
different tiles displaced into range). After full-pipeline display, no
geographic hotspot structure should remain.

In [5]:
def gi_star(xy, y, threshold=THRESHOLD_M, n_perm=499):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w = libpysal.weights.DistanceBand.from_array(
            xy, threshold=threshold, binary=True, silence_warnings=True)
        g = esda.G_Local(y, w, star=True, permutations=n_perm)
    return g

g_orig = gi_star(xy_orig, y_death)
g_jit  = gi_star(xy_jit,  y_death)
g_full = gi_star(xy_full, y_death)

hot_orig = (g_orig.p_sim < 0.05) & (g_orig.Zs > 0)
hot_jit  = (g_jit.p_sim  < 0.05) & (g_jit.Zs  > 0)
hot_full = (g_full.p_sim < 0.05) & (g_full.Zs  > 0)

print('Significant hotspot records (p < 0.05, positive Gi* z-score):')
print(f'  Original     : {hot_orig.sum():3d} / {len(deaths)}')
print(f'  Jitter-only  : {hot_jit.sum():3d} / {len(deaths)}')
print(f'  Full pipeline: {hot_full.sum():3d} / {len(deaths)}')
print()
# Persistence: hotspots that survive jitter displacement
persist = (hot_orig & hot_jit).sum()
print(f'Hotspots persistent under jitter: {persist} / {hot_orig.sum()} ({100*persist/max(hot_orig.sum(),1):.0f}%)')

('WARNING: ', 183, ' is an island (no neighbors)')
('WARNING: ', 248, ' is an island (no neighbors)')
Significant hotspot records (p < 0.05, positive Gi* z-score):
  Original     :  37 / 250
  Jitter-only  :  37 / 250
  Full pipeline:   7 / 250

Hotspots persistent under jitter: 6 / 37 (16%)


In [6]:
# Folium map: hotspot persistence
m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15,
               tiles='cartodbpositron')

for i, (_, r) in enumerate(deaths.iterrows()):
    orig_hot = hot_orig[i]
    jit_hot  = hot_jit[i]
    if orig_hot and jit_hot:
        colour, label = '#cc0000', 'Hotspot: persists under jitter'
    elif orig_hot:
        colour, label = '#ff8800', 'Hotspot: lost under jitter'
    elif jit_hot:
        colour, label = '#0055cc', 'Hotspot: gained under jitter'
    else:
        colour, label = '#aaaaaa', 'Not a hotspot'
    folium.CircleMarker(
        location=[r.LAT, r.LON], radius=5,
        color=colour, fill=True, fill_color=colour, fill_opacity=0.75,
        tooltip=f"FID {r.FID} — {label}"
    ).add_to(m)

for _, p in pumps.iterrows():
    folium.CircleMarker(
        location=[p.LAT, p.LON], radius=7,
        color='black', fill=True, fill_color='blue', fill_opacity=1.0,
        tooltip=p.Street
    ).add_to(m)

m

## Summary

| Metric | What it captures | Jitter-only result | Full pipeline result |
|--------|------------------|--------------------|----------------------|
| Ripley's K AUC | Clustering intensity across scales | High preservation | Near-zero (PRP destroys) |
| Moran's I | Global spatial autocorrelation of counts | Similar I, similar p | Uncorrelated |
| Getis-Ord Gi* hotspots | Local high-count clusters | Most persist | None identifiable |

**Key insight:** The jitter displacement mechanism (±62.5 m) preserves most
spatial structure — useful for analysts with decode access. The PRP tile shuffle
is the layer that destroys spatial structure for the display tier, providing
strong structural privacy even without the AEAD encryption.

Part 2 (NB13) continues with KDE surface fidelity, multi-scale evaluation
across jitter levels, the privacy–utility frontier curve, and failure cases.

## References

- Lin, Y. (2023). Geo-indistinguishable masking: enhancing privacy protection in
  spatial point mapping. *Cartography and Geographic Information Science.*
  https://doi.org/10.1080/15230406.2023.2267967

- Ripley, B. D. (1976). The second-order analysis of stationary point processes.
  *Journal of Applied Probability, 13*(2), 255–266.
  https://doi.org/10.2307/3212829

- Moran, P. A. P. (1950). Notes on continuous stochastic phenomena.
  *Biometrika, 37*(1–2), 17–23. https://doi.org/10.2307/2332142

- Getis, A., & Ord, J. K. (1992). The analysis of spatial association by use
  of distance statistics. *Geographical Analysis, 24*(3), 189–206.
  https://doi.org/10.1111/j.1538-4632.1992.tb00261.x

- Rey, S. J., & Anselin, L. (2007). PySAL: A Python library of spatial
  analytical methods. *The Review of Regional Studies, 37*(1), 5–27.

- Snow, J. (1855). *On the Mode of Communication of Cholera* (2nd ed.). John Churchill.

## Glossary

| Term | Definition |
|------|------------|
| Ripley's K(t) | Expected number of additional points within distance t of a random point, normalised by density |
| L-function | Variance-stabilised transform of K: L(t) = √(K(t)/π) − t; equals 0 under CSR |
| CSR | Complete Spatial Randomness — the Poisson process null model for point patterns |
| Moran's I | Global index of spatial autocorrelation; ranges from −1 (dispersed) to +1 (clustered) |
| spatial weights | Matrix w_{ij} encoding neighbourhood structure; here: 1 if distance < 400 m |
| Getis-Ord Gi* | Local statistic identifying records in statistically significant high-value clusters |
| hotspot | Cluster of high attribute values that is statistically significant at p < 0.05 |
| jitter-only | Displacement of coordinates by ±J m without PRP tile shuffle; isolates noise effect |
| full pipeline | Complete encode → render_coordinates output: PRP shuffle + jitter |